# Ana for EPB

In [ ]:
#regular packages
import pandas as pd
import string
import numpy as np
import os

#nlp packages
import nltk
import contractions
from nltk.corpus import stopwords
#nltk.download('wordnet')
from nltk.stem import WordNetLemmatizer, SnowballStemmer
from nltk.stem.porter import *

from bertopic import BERTopic

In [ ]:
#import nltk
#nltk.download('punkt')
#nltk.download('stopwords')
#import ssl

#try:
#    _create_unverified_https_context = ssl._create_unverified_context
#except AttributeError:
#    pass
#else:
#    ssl._create_default_https_context = _create_unverified_https_context#

#nltk.download()

# 0 Load Data
## 0.1 Individual decade

In [ ]:
df70 = pd.read_csv('data_papers/1970s.csv')
df80 = pd.read_csv('data_papers/1980s.csv')
df90 = pd.read_csv('data_papers/1990s.csv')
df00 = pd.read_csv('data_papers/2000s.csv')
df10 = pd.read_csv('data_papers/2010s.csv')
df20 = pd.read_csv('data_papers/2020s.csv')

df_inpress = pd.read_csv('data_papers/inpress.csv')
df_data = pd.read_csv('data_papers/EPB_Data.csv')
df_fg = pd.read_csv('data_papers/EPB_FG.csv')

In [ ]:
len(df80)

In [ ]:
df_20 = pd.concat([df20, df_inpress, df_data, df_fg],  ignore_index=True)

In [ ]:
directory_path = 'data_papers'
# List all files in the directory
files = os.listdir(directory_path)

# Filter out only CSV files
csv_files = [file for file in files if file.endswith('.csv')]

full_df = pd.DataFrame()
# Iterate through each CSV file and read it using pandas
for csv_file in csv_files:
    file_path = os.path.join(directory_path, csv_file)
    df = pd.read_csv(file_path)
    full_df = pd.concat([full_df, df])
    
    # Now you can work with the dataframe 'df'
    # For example, print the first few rows
    print(f"Contents of {csv_file}:")
print(len(full_df))
#print(full_df.head())    

In [ ]:
rows_with_nan = full_df[full_df["Abstract Note"] == 'nan']
rows_with_nan

## 1 Preprocessing: Cleaning Text 

### 1.1 Removing stop words along with other words
* remove the words shows uo more than 1000 time 
* remove meaingless words 

In [ ]:
regular_stop_words = stopwords.words('english')

In [ ]:
words_1k = ['urban', 'model', 'city', 'planning', 'study', 'data', 'spatial', 'use', 'area', 'based', 'paper', 'system']

words_other = ['result', 'ha', 'method', 'approach', 'using', 'different', 'new', 'used', 'level', 'research', 'two', 'information', 'wa', 'also', 'problem', 'effect'
                'type', 'show', 'case', 'impact', 'within', 'however', 'developed', 'application', 'high', 'context', 'relationship', 'may', 'potential', 'characteristic', 'finding', 'term', 'well', 'value', 'first', 'present', 'number', 'large', 'distribution', "one", "two", "three", "four", "five", "six", "seven", "eight", "nine", "ten", "finding", "built"]

additional_words = ['provide', 'technique', 'important', 'various', 'existing', 'way', 'applied', 'understanding', 'significant', 'need', 'future', 'many', 'size', 'found', 'across', 'higher', 'function', 'identify', 'presented', 'time', 'issue', 'made', 'user', 'work', 'author', 'mean', 'map', 'article']

epb_stop_words = regular_stop_words + words_1k + words_other + additional_words

## 1.2 Functions 

In [ ]:
def preprocess(text_col):
    
    #lowe cases
    text_col = text_col.apply(lambda x: ' '.join([w.lower() for w in x.split()]))
    # expand contractions  
    text_col = text_col.apply(lambda x: ' '.join([contractions.fix(word) for word in x.split()]))
    # remove numbers
    text_col = text_col.apply(lambda x: ' '.join(re.sub("[^a-zA-Z]+", " ", x).split()))
    
    return text_col

In [ ]:
def remove_plurals_stem(text):
    special_words = {"gis"}
    lemmatizer = WordNetLemmatizer()
    words = text.split()
    processed_words = []
    for word in words:
        if word in special_words:
            processed_words.append(word)
        else:
            stem = lemmatizer.lemmatize(word)
            if stem not in processed_words:
                processed_words.append(stem)
    
    return ' '.join(processed_words)
    
def remove_plurals(text):
    special_words = {"gis"}
    lemmatizer = WordNetLemmatizer()
    words = text.split()
    processed_words = [word if word in special_words else lemmatizer.lemmatize(word) for word in words]
    return ' '.join(processed_words)
     #= [word if word in special_words else lemmatizer.lemmatize(word) for word in words]
    
    #return ' '.join(processed_words)

def remove_stopwords(text_col, stopwords):
    # remove stopwords
    text_col = text_col.apply(lambda x: ' '.join([w for w in x.split() if w not in stopwords]))
    return text_col

In [ ]:
def remove_punctuation(text_col):
    special_chars  = {'-'}
    punctuation = ''.join(c for c in string.punctuation if c not in special_chars)
    # Define the set of punctuation characters
    #punctuation = set(string.punctuation)
    # Remove punctuation except for special characters
    text_col = text_col.apply(lambda x: ''.join([i for i in x if i not in punctuation]))
    
    return text_col

In [ ]:
def __get_word_freq(data_col):
    word_freq = data_col.str.split(expand=True).stack().value_counts()
    #print(type(word_freq))
    return word_freq

In [ ]:
def __pre_process__(data, stopword):
    #print(data.head())
    data['Abstract Note'] = data['Abstract Note'].astype(str)
    data['abstract'] = preprocess(data['Abstract Note'])
    data['abstract'] = remove_punctuation(data['abstract'])
    data['abstract'] = data['abstract'].apply(lambda x: remove_plurals(x))
    data['abstract'] = remove_stopwords(data['abstract'], stopword)
    data_freq = __get_word_freq(data['abstract'])
    return data_freq#[:30]

## 1.3 Full Test Cleaning

In [ ]:
full_df['Abstract Note'] = full_df['Abstract Note'].astype(str)
full_df['abstract'] = preprocess(full_df['Abstract Note'])
full_df['abstract'] = full_df['abstract'].apply(lambda x: remove_plurals(x))
#full_df['abstract'] = remove_stopwords(full_df['abstract'], epb_stop_words)
full_df['abstract'] = remove_stopwords(full_df['abstract'], regular_stop_words)
full_freq = __get_word_freq(full_df['abstract'])

In [ ]:
full_freq

## 2 Yearly Stat

### 2.1 paper published each year

In [ ]:
#Pre-processing
#df['year'] = df['year'].fillna(0)
full_df['Publication Year'] = full_df['Publication Year'].astype(int)
full_df['amount'] = 1

In [ ]:
df = full_df.loc[:, ["Publication Year", "amount"]]

In [ ]:
df.head()

In [ ]:
#display histogram of yearly news amount
def DisplayNewsYearly(data):
    import matplotlib.pyplot as plt
    %matplotlib inline
    #fig = plt.figure()
    plt.figure(figsize=(25,10))
    #plt.style.use('ggplot')

    #x = list(data.year)
    x = list(data['Publication Year'])
    y = list(data.amount)
    x_pos = [i for i, _ in enumerate(x)]

    #'#469EB4', '#4E62AB'
    #plt.bar(x_pos, y, width=0.80, color='#43a2ca')
    plt.bar(x_pos, y, width=0.80, color='#469EB4')
    plt.xlabel("Year", fontsize = 24, fontname = "Arial")
    plt.ylabel("Paper Count", fontsize = 24, fontname = "Arial")

    plt.xticks(x_pos, x)

    plt.show()

In [ ]:
yearcount = df.groupby(by=["Publication Year"]).sum().reset_index()
yearcount.loc[:, ['Publication Year', 'amount']].head()

In [ ]:
DisplayNewsYearly(yearcount.loc[:, ['Publication Year', 'amount']])

### 2.2 AVG Authorship by year

## 3 Topic Modeling 

In [ ]:
#df['abstract'] = preprocess(df['Abstract Note'], )
#full_df['title_en'] = preprocess(full_df['Title'], epb_stop_words)

In [ ]:
#full_df['abstract'] = remove_stopwords(full_df['abstract'], epb_stop_words)

In [ ]:
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer

from bertopic import BERTopic
from bertopic.vectorizers import ClassTfidfTransformer
from bertopic.representation import KeyBERTInspired

In [ ]:
#Documents for the inputs
#docs = list(full_df.loc[:,"abstract"].values)
docs = list(full_df.loc[:,"abstract"].values)
print("The length of the data is", len(docs))

In [ ]:
#Sub-models
#1, Document Embedding
doc_embedding_model = SentenceTransformer("all-mpnet-base-v2") #high performance
#doc_embedding_model = SentenceTransformer("all-MiniLM-L6-v2")


#2, Dimensionality Reduction
dimensionality_reduction = UMAP(n_neighbors = 15, n_components = 5,
                  min_dist=0.0, metric='cosine', random_state = 68)#generate consistent results

#3, Document Clustering
doc_clustering_model = HDBSCAN(min_cluster_size = 10, metric= 'euclidean', cluster_selection_method='eom', prediction_data=True)
#doc_clustering_model = HDBSCAN(min_cluster_size=10, metric= 'euclidean', cluster_selection_method='eom', prediction_data=True)

#4, Topic representation: fine-tuning the topic representation
#4.1 CountVectorizer
#vectorizer_model_tuning = CountVectorizer(stop_words="english", min_df = 5)
vectorizer_model_tuning = CountVectorizer(ngram_range = (1,2), stop_words= epb_stop_words, min_df = 5)

#4.2 TF-IDF
#ctfidf_model = ClassTfidfTransformer(reduce_frequent_words=True, bm25_weighting=True)
#ctfidf_model = ClassTfidfTransformer(bm25_weighting=True)
ctfidf_model = ClassTfidfTransformer(reduce_frequent_words=True)
#ctfidf_model = ClassTfidfTransformer()

# Step 6 - (Optional) Fine-tune topic representations with
# a `bertopic.representation` model
#representation_model = KeyBERTInspired()

### 2.1.2 Results Tuning
#### 2.1.2.1 No Tuning

In [ ]:
#Application topic model
topic_model_nt = BERTopic(embedding_model = doc_embedding_model, #1
                       umap_model = dimensionality_reduction,#2
                       hdbscan_model = doc_clustering_model,#3
                       top_n_words = 15, min_topic_size=15, nr_topics='auto',
                       language = "english", calculate_probabilities = False, verbose = True)

In [ ]:
topics, probs= topic_model_nt.fit_transform(docs)

In [ ]:
freq1 = topic_model_nt.get_topic_info()#good
freq1

#### 2.1.2.2 Only CV on Non Tuning model
Apply CV after Get Results

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

#vectorizer_model_tuning_after = CountVectorizer(ngram_range = (1,2), stop_words="english", min_df = 5)
vectorizer_model_tuning_after = CountVectorizer(ngram_range = (1,2), stop_words= epb_stop_words, min_df = 5)

topic_model_nt.update_topics(docs, vectorizer_model = vectorizer_model_tuning_after)

In [ ]:
#No tuning
freq2 = topic_model_nt.get_topic_info()#good
freq2

#### 2.1.2.2Apply CV When Get Results

In [ ]:
#Application topic model
topic_model_cv = BERTopic(embedding_model = doc_embedding_model, #1
                       umap_model = dimensionality_reduction,#2
                       hdbscan_model = doc_clustering_model,#3
                       vectorizer_model = vectorizer_model_tuning,#4.1
                       top_n_words = 15, min_topic_size=15, nr_topics='auto',
                       language = "english", calculate_probabilities = False, verbose = True)

In [ ]:
topics_2, probs= topic_model_cv.fit_transform(docs)

In [ ]:
freq3 = topic_model_cv.get_topic_info()#good
freq3

#### 2.1.2.3 Only TF-IDF (Used this one for editoral)

In [ ]:
#Application topic model
topic_model_tfidf = BERTopic(embedding_model = doc_embedding_model, #1
                       umap_model = dimensionality_reduction,#2
                       hdbscan_model = doc_clustering_model,#3
                       ctfidf_model = ctfidf_model,#4.2
                       top_n_words = 15, min_topic_size=15, nr_topics='auto',
                       language = "english", calculate_probabilities = False, verbose = True)

In [ ]:
topics_3, probs= topic_model_tfidf.fit_transform(docs)

In [ ]:
freq4 = topic_model_tfidf.get_topic_info()#good use this one 
freq4

In [ ]:

# create a data frame with topic number along with its freq and all words
def _get_topic_allwords_(all_topics, t_model):
    topic_list = []
    freq_list = []
    words_list = []

    for topic in all_topics:
        topic_name = "Topic" + str(topic)
        topic_list.append(topic_name)
        # print(topic_name)
        frq = t_model.get_topic_freq(topic)
        freq_list.append(frq)
        wordset = t_model.get_topic(topic)[:15]
        # print("====Get item from get_topic Results====")
        topic_word = []
        for item in wordset:
            # print()
            # topic_word += '' + str(item[0])
            topic_word.append(item[0])

        words_list.append(topic_word)
    # words_list.append(join(topic_word))
    topics_df = pd.DataFrame(list(zip(topic_list, freq_list, words_list)), columns=['Topic', 'Freq', 'Words'])
    return topics_df

In [ ]:
tfidf_topic = _get_topic_allwords_(topics_3, topic_model_tfidf)

In [ ]:
tfidf_topic.to_csv('results/topics_tfidf_model.csv')

In [ ]:
#save model
#1,nonfliter; 2,withseed; 3,hperform; 4,remove stop word; 5, auto_drop
topic_model_tfidf.save("results/model_tfidf_model")
#save topics to list
df = pd.DataFrame(data={"col1": topics_3})
df.to_csv("results/topics_from_tfidf_model.csv", sep=',',index=False)

#### 2.1.2.3 TF-IDF + CV
Apply CV after

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
vectorizer_model_tuning_after = CountVectorizer(ngram_range = (1,2), stop_words= epb_stop_words, min_df = 5)
#topic_model_nt.update_topics(docs, vectorizer_model = vectorizer_model_tuning_after)
topic_model_tfidf.update_topics(docs, vectorizer_model = vectorizer_model_tuning_after )

In [ ]:
freq5 = topic_model_tfidf.get_topic_info()#good
freq5

#### 2.1.2.4 Apply CV and TF-IDF at the same time

In [ ]:
#Application topic model
topic_model_cv_tfidf = BERTopic(embedding_model = doc_embedding_model, #1
                       umap_model = dimensionality_reduction,#2
                       hdbscan_model = doc_clustering_model,#3
                       vectorizer_model = vectorizer_model_tuning,#4.1
                       ctfidf_model = ctfidf_model,#4.2
                       top_n_words = 15, min_topic_size=15, nr_topics='auto',
                       language = "english", calculate_probabilities = False, verbose = True)

In [ ]:
topics_4, probs= topic_model_cv_tfidf.fit_transform(docs)

In [ ]:
freq6 = topic_model_cv_tfidf.get_topic_info()
freq6

## 3.2 Topic Overtime

In [ ]:
#Create list contains all year
full_df['year'] = full_df['Publication Year'].astype(int)
timestamps = full_df['Publication Year'].to_list()
len(timestamps)

In [ ]:
topics_over_time = topic_model_tfidf.topics_over_time(docs, timestamps, nr_bins=None,
                                                datetime_format=None,
                                                evolution_tuning=True,
                                                global_tuning=True)

In [ ]:
topic_model_tfidf.visualize_topics_over_time(topics_over_time)

In [ ]:
topics_over_time.head()

In [ ]:
topics_over_time.to_csv("results/topics_over_time_tfidf_model.csv")

In [ ]:
tf_ft_topic = _get_topic_allwords_(freq5.Topic, topic_model_tfidf)
#freq1.Topic, topic_model_nt
tf_ft_topic.to_csv('results/topics_tf_ft_topic.csv')